# Day 2 v3 — Few-step Self-Distillation (Kaggle T4)

**The pivot:** instead of fine-tuning, we run **self-distillation**. Your Day 1 checkpoint is frozen as a TEACHER and runs 5 DDIM substeps to produce refined targets. A trainable STUDENT (initialized from the same checkpoint) learns to match the teacher's 5-substep refinement in a SINGLE forward pass. After training, the student is plugged back into the standard 5-step DDIM sampler — each step is now distilled to be much higher quality.

**Why this should work where fine-tuning didn't:**
- Fine-tuning was drifting the model away from its known-good operating point with no clear target.
- Distillation gives the student a *concrete, refined* target to imitate. The teacher's 5-substep DDIM gives a strictly more accurate single-shot prediction than the student's untrained single shot at the same noise level.
- Citable foundations: Salimans & Ho 2022 (Progressive Distillation), Song et al. 2023 (Consistency Models). This is a real architectural/training contribution.

**Wall time on T4: ~50–70 minutes total** (smoke test + 20-epoch distill).

**Before running:** push the new files to GitHub:
- `train_distill.py`
- updated `train.py` (already pushed last round)
- updated `diffusion.py` (already pushed last round)
- `kaggle_path_discovery.py` (already pushed)

In [ ]:
# === Cell 1: install ===
!pip install -q --upgrade pip
!pip install -q scikit-image lpips matplotlib

In [ ]:
# === Cell 2: clone repo ===
import os, sys, shutil, subprocess
BRANCH = "main"
REPO_URL = "https://github.com/chirana07/Diffusion_new_final.git"
REPO_DIR = "/kaggle/working/Diffunet"
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

for f in ("train_distill.py", "kaggle_path_discovery.py"):
    if not os.path.exists(f):
        raise SystemExit(f"{f} missing in cloned repo. Push your latest commits first.")
print("Repo ready.")

In [ ]:
# === Cell 3: PREFLIGHT — discover datasets, set up canonical layout ===
import kaggle_path_discovery as kpd
kpd.diagnose("/kaggle/input")
discoveries = kpd.discover_all("/kaggle/input")
kpd.print_picks(discoveries)

# === MANUAL OVERRIDE (uncomment + edit if discovery picked wrong) ===
# discoveries['lolv2_real_train'] = {'low_dir': '...', 'high_dir': '...', 'n_images': 689, 'kind': 'train'}
# discoveries['lolv2_real_test']  = {'low_dir': '...', 'high_dir': '...', 'n_images': 100, 'kind': 'test'}

if not discoveries.get('lolv2_real_train'):
    raise SystemExit("LOL-v2 Real TRAIN split not found. Re-attach dataset or set the path manually above.")
if not discoveries.get('lolv2_real_test'):
    raise SystemExit("LOL-v2 Real TEST split not found.")

DATASET_ROOT = kpd.setup_canonical_symlinks(discoveries, working_root="/kaggle/working")
print(f"\nCanonical layout at: {DATASET_ROOT}")
counts = kpd.validate_canonical(DATASET_ROOT, require_train=True, require_test=True,
                                 min_train_images=300, min_test_images=15)
for k, v in counts.items(): print(f"  {k:<28s}  {v}")

In [ ]:
# === Cell 4: locate the Day 1 checkpoint (the one we distill FROM) ===
import glob
ckpt_hits = sorted(glob.glob("/kaggle/input/**/*.pth", recursive=True))
preferred = [c for c in ckpt_hits if "final" in c.lower()]
if not preferred:
    preferred = [c for c in ckpt_hits if any(k in c.lower() for k in ("best", "last"))]
preferred = [c for c in preferred if "distill" not in c.lower() and "_ft" not in c.lower()]
INIT_FROM = preferred[0] if preferred else (ckpt_hits[0] if ckpt_hits else None)

# === MANUAL OVERRIDE ===
# INIT_FROM = "/kaggle/input/lumidiff-checkpoint/final.pth"

if INIT_FROM is None:
    raise SystemExit("No .pth checkpoint found under /kaggle/input/. Attach your Day 1 checkpoint as a Kaggle dataset.")
print(f"INIT_FROM = {INIT_FROM}")
print(f"Size: {os.path.getsize(INIT_FROM) / 1e6:.1f} MB")

In [ ]:
# === Cell 5: GPU sanity ===
import torch
print("cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - set Accelerator -> GPU T4")
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Settings -> Accelerator -> GPU T4 -> Save, then re-run.")

In [ ]:
# === Cell 6: SMOKE TEST — 1 epoch + 5 val batches (~3-4 min) ===
# Catches: bad checkpoint loading, dataset path issues, GPU OOM, teacher refinement bugs.
# ALWAYS run before the long distillation.
import sys, subprocess
cmd = [
    sys.executable, "train_distill.py",
    "--layout", "lolv2_real",
    "--dataset-root", DATASET_ROOT,
    "--init-from", INIT_FROM,
    "--epochs", "1",
    "--teacher-steps", "5",
    "--student-inference-steps", "5",
    "--lr", "1e-5",
    "--w-distill", "1.0",
    "--w-anchor", "0.5",
    "--batch-size", "4",
    "--crop-size", "256",
    "--num-workers", "2",
    "--val-every", "1",
    "--val-max-batches", "5",
    "--tag", "smoke_distill",
]
print("$ " + " ".join(cmd))
subprocess.run(cmd, check=True)
print("\nSmoke test passed. Proceed to Cell 7.")

In [ ]:
# === Cell 7: Distillation hyperparameters (edit if needed) ===
DISTILL_EPOCHS  = 20      # 15-25 reasonable; 20 is the sweet spot
DISTILL_LR      = "5e-5"  # higher than fine-tuning since we have a clear target
TEACHER_STEPS   = 5       # DDIM sub-steps the teacher takes (more = better target, slower)
STUDENT_STEPS   = 5       # student inference step count (validation uses this)
W_DISTILL       = "1.0"   # weight on char(student_x0, teacher_x0)
W_ANCHOR        = "0.5"   # weight on char(student_x0, gt_residual)
BATCH           = 4
CROP            = 256
VAL_EVERY       = 5

print("Distillation config:")
print(f"  epochs           = {DISTILL_EPOCHS}")
print(f"  lr               = {DISTILL_LR}")
print(f"  teacher_steps    = {TEACHER_STEPS}")
print(f"  student_steps    = {STUDENT_STEPS}")
print(f"  w_distill        = {W_DISTILL}")
print(f"  w_anchor         = {W_ANCHOR}")
print(f"  val_every        = {VAL_EVERY}")

In [ ]:
# === Cell 8: RUN — full distillation (~50-70 min on T4) ===
# Output: ./checkpoints/best_distill_K5.pth (saved each time 5-step val PSNR improves)
cmd = [
    sys.executable, "train_distill.py",
    "--layout", "lolv2_real",
    "--dataset-root", DATASET_ROOT,
    "--init-from", INIT_FROM,
    "--epochs", str(DISTILL_EPOCHS),
    "--teacher-steps", str(TEACHER_STEPS),
    "--student-inference-steps", str(STUDENT_STEPS),
    "--lr", DISTILL_LR,
    "--w-distill", W_DISTILL,
    "--w-anchor", W_ANCHOR,
    "--batch-size", str(BATCH),
    "--crop-size", str(CROP),
    "--num-workers", "2",
    "--val-every", str(VAL_EVERY),
    "--tag", "distill_K5",
]
print("$ " + " ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# === Cell 9: collect outputs and print val_psnr trajectory ===
import shutil, glob, csv
OUT = "/kaggle/working/day2_distill"
os.makedirs(OUT, exist_ok=True)

for src in glob.glob("./checkpoints/best_distill_K5.pth") + \
           glob.glob("./checkpoints/last_distill_K5.pth") + \
           glob.glob("./checkpoints/distill_log_distill_K5.csv"):
    shutil.copy2(src, OUT)
    print("copied", src)

for log in sorted(glob.glob(os.path.join(OUT, "distill_log_*.csv"))):
    print(f"\n--- {os.path.basename(log)} ---")
    rows = list(csv.reader(open(log)))
    header = rows[0]
    val_psnr_idx = header.index("val_psnr_5step")
    val_ssim_idx = header.index("val_ssim_5step")
    loss_idx     = header.index("loss_total")
    print("epoch  loss_total  val_psnr_5step  val_ssim_5step")
    for r in rows[1:]:
        if r[val_psnr_idx] not in ("-1.0", ""):
            print(f"{r[0]:<5}  {float(r[loss_idx]):>9.4f}  {float(r[val_psnr_idx]):>13.3f}  {float(r[val_ssim_idx]):.4f}")
        else:
            print(f"{r[0]:<5}  {float(r[loss_idx]):>9.4f}  (no val)")

out_zip = "/kaggle/working/day2_distill.zip"
if os.path.exists(out_zip): os.remove(out_zip)
shutil.make_archive(out_zip.replace(".zip", ""), "zip", OUT)
print("\nWrote", out_zip)
print("\nNext: Output tab -> download day2_distill.zip")
print("Then upload best_distill_K5.pth as a NEW Kaggle dataset 'lumidiff-distill-ckpt'.")
print("Then run kaggle_eval_illum.ipynb (eval works against any of the saved checkpoints).")